#Bronze to Silver: Data Cleaning and Transformation for Dimension Tables

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType
catalog_name = 'ecommerce'

# Brands

In [0]:
df_bronze = spark.table(f'{catalog_name}.bronze.brz_brands')
df_bronze.show(10)

In [0]:
df_silver = df_bronze.withColumn("brand_name", F.trim(F.col("brand_name")))
df_silver.show(10)


In [0]:
df_silver = df_silver.withColumn("brand_code",  F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', ''))
df_silver.show(10)

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
# Anamolies Detection
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}
#Pyspark replace is easy
df_silver = df_silver.replace(to_replace=anomalies, subset=["category_code"])
df_silver.select("category_code").distinct().show()


In [0]:
#Write data to the silver layer (catalog: ecommerce, schema: silver, table: slv_brands)
df_silver.write.mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", "True") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")


## Category

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
df_bronze.show(10)

In [0]:
df_duplicates = df_bronze.groupBy("category_code").count().filter(F.col("count")>1)
display(df_duplicates)
                                                                        

In [0]:
df_silver = df_bronze.dropDuplicates(["category_code"])
display(df_silver)

In [0]:
df_silver = df_silver.withColumn("category_code", F.upper(F.col("category_code")))
display(df_silver)

In [0]:
# write Raw data from bronze to silver layer (catalog: Ecommerce, schema: silver, table: slv_category)
df_silver.write.mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", "True") \
        .saveAsTable(f"{catalog_name}.silver.slv_category")


## Products

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_products)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_products")
# Get row & coloumn count 
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

In [0]:
display(df_bronze.limit(5))

In [0]:
# Check weight_grams column
df_bronze.select("weight_grams").show(5, truncate=False)


In [0]:
# Replace 'g' with ''
df_silver = df_bronze.withColumn(
    "weight_grams",
     F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType())
)
df_silver.select("weight_grams").show(5, truncate=False)

In [0]:
# Check_length_cm (comma instead of dot)
df_bronze.select("length_cm").show(3)

In [0]:
# Replace , with .
df_silver = df_bronze.withColumn(
    "length_cm",
    F.regexp_replace(F.col("length_cm"), ",", ".")
    .cast(FloatType())    
)
df_silver.select("length_cm").show(3)


In [0]:
# Category code & brand code is in lower case. we need to make it all uppercase
# df_silver.select("brand_code","category_code").show(2)
df_silver =df_silver.withColumn(
    "category_code", 
    F.upper(F.col("category_code"))
).withColumn(
    "brand_code",
    F.upper(F.col("brand_code"))
)
df_silver.select("brand_code","category_code").show(5)
    

Spelling mistake in material column

In [0]:
df_silver.select('material').distinct().show(truncate=False)

In [0]:
# Fix spelling mistakes
df_silver = df_silver.withColumn(
    "material",
    F.when(F.col("material") == 'Coton', 'cotton')
    .when(F.col("material") == "Alumium", 'Aluminum')
    .when(F.col('material') == 'Ruber','Rubber')
    .otherwise(F.col("material"))
)
df_silver.select('material').distinct().show()

In [0]:
# Negative values in rating_count
df_silver.filter(F.col('rating_count')<0).select('rating_count').show(5)

In [0]:
# Convert negative rating_count to positive
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col('rating_count')))
    .otherwise(F.lit(0)) # if null replace with 0
)

In [0]:
# Check the final cleaned data 
df_silver.select (
    "weight_grams",
    "length_cm",
    "brand_code",
    "category_code",
    "material",
    "rating_count"
).show(10, truncate=False)

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_dim_products)
df_silver.write.mode("overwrite").format("delta")\
    .option('mergeSchema', 'true') \
    .saveAsTable(f"{catalog_name}.silver.slv_products")
    

## Customers

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_customer)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_customers")
# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")


In [0]:
df_bronze.show(10)

In [0]:
# Handling null values in customer_id column
null_count = df_bronze.filter(F.col('customer_id').isNull()).count()
null_count

In [0]:
# There are 300 null values in customer_id column. Display some of those
df_bronze.filter(F.col('customer_id').isNull()).show(5)

In [0]:
# Drop rows where 'customer_id' is null
df_silver = df_bronze.dropna(subset=['customer_id'])
# Get the row count
row_count = df_silver.count()
print(f"Row count after dropping null values: {row_count}")

In [0]:
# Handle NULL values in phone column
null_count = df_silver.filter(F.col('phone').isNull()).count()
print(f'Number of nulls in phone {null_count}')

In [0]:
### Fill null values with 'Not Available'
df_silver = df_silver.fillna('Not Available', subset=['phone'])
# # sanity check (If any nulls still exist)
df_silver.filter(F.col('phone').isNull()).show()

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_customers)
df_silver.write.mode("overwrite") \
    .format('delta') \
    .option('mergeSchema', 'true') \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

## Calendar/Date

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calendar")
# Get row and column count
row_count, column_count = df_bronze.count(),len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")
df_bronze.show(10)
#)

In [0]:
df_bronze.printSchema()

In [0]:
# Converting String to Date
from pyspark.sql.functions import to_date
df_silver = df_bronze.withColumn('date', to_date(df_bronze['date'],"dd-MM-yyyy"))
df_silver.show(10)

In [0]:
# Find duplicate rows in the DataFrame
duplicates = df_silver.groupBy('date').count().filter('count > 1')
# Show the duplicate rows
print(f'Total duplicated rows: {duplicates.count()}')
display(duplicates)

In [0]:
# Remove duplicate rows
df_silver = df_silver.dropDuplicates(['date'])

# Get row count
row_count = df_silver.count()

print("Rows After removing Duplicates: ", row_count)

In [0]:
# day_name normalize casing
# Capitalize first letter of each word in day_name
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))
df_silver.show(5)

In [0]:
# Convert negative week_of_year to positive
df_silver = df_silver.withColumn("week_of_year", F.abs(F.col("week_of_year"))) # conver negative to positive
df_silver.show(5)

In [0]:
# Enhance quarter and week_of_year column
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))
df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))

df_silver.show(3)

In [0]:
# Rename columns
df_silver = df_silver.withColumnRenamed("week_of_year", "week")



In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_calendar)
df_silver.write.mode('overwrite')  \
    .option("mergeSchema", "true") \
.saveAsTable(f"{catalog_name}.silver.slv_calendar")
                                              